# **0. LIBRERÍAS Y CONFIGURACIÓN**

In [34]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.impute import KNNImputer

# Preprocesamiento y Modelado
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.preprocessing import TargetEncoder
# modelos

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

# Métricas
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# **1. Carga de datos procesados y exploración inicial**

In [35]:
df = pd.read_parquet("../data/processed/dataset.parquet")

# El parquet ya viene con el filtro de precio aplicado desde el EDA.
# Aquí reforzamos solo el filtro de áreas para evitar outliers extremos.
df_filtrado = df[
    (df["Área Construida (m2)"] <= 2500) &
    (df["Área Privada (m2)"] <= 2500)
].copy()

print(df_filtrado["Área Construida (m2)"].max())
print(df_filtrado["Área Privada (m2)"].max())

2416.0
2416.0


# **Feature engineering**

# **Split de datos (train/test)**

### Separación de variables predictoras (X) y variable objetivo (y, "Precio")

### <span style="color: #dc2626;">!!! **Importante: comentarios para recordar**</span>


1. voy a usar Barrio_group como parte de las features, pero la idea es que luego podamos definir realmente cuales van a ser las variables que vamos a usar despues de que toda la limpieza y análisis haya terminado, estoy usando el que hace que vaya a "otros"
2. también estoy usando piso_cat
3. pareciera que para poder usar optuna, es mejor hacer algo como train, test y validation porque optuna corremos el riesgo de ajustar el modelo al test indirectamente, la idea es que  Optuna compare configuraciones sin “mirar” el test final, y eso es algo que no hicimos en el trabajo de la profe camila

70% train: el modelo aprende
15% validation : Optuna prueba distintas combinaciones y decide cuáles son mejores
15% test: una sola vez al final para medir el desempeño real

4. tenemos que definir que métrica nos importa mas para nuestros modelos entre MAE, RMSE y R2
5. tampoco podemos incluir area privada m2 porque hace multicolinealidad


In [38]:
df_filtrado.columns

Index(['ID', 'Barrio', 'Tipo de Inmueble', 'Estado', 'Antigüedad',
       'Área Construida (m2)', 'Área Privada (m2)', 'Estrato', 'Baños',
       'Habitaciones', 'Parqueaderos', 'Piso N°', 'URL', 'Precio',
       'Barrio_clean', 'Barrio_group', 'Piso_cat'],
      dtype='object')

In [39]:

numeric_features = [
    "Área Construida (m2)",
    "Estrato",
    "Baños",
    "Habitaciones",
    "Parqueaderos",
]

categorical_features = [
    "Tipo de Inmueble",
    "Estado",
    "Antigüedad",
    "Barrio_group",
    "Piso_cat",
]

features = numeric_features + categorical_features



In [40]:
# En el EDA vimos que estas son las variables que tienen valores faltantes, por lo que las vamos a imputar con KNN Imputer.

cols_int   = ["Baños", "Habitaciones"]
cols_float = ["Área Construida (m2)"]
cols_all   = cols_int + cols_float

# Separación de variables predictoras (X) y variable objetivo (y, "Precio")

X = df_filtrado[features].copy()
y = df_filtrado["Precio"].copy()

# SPlit antes de imputar con knn las variables que nos hacen falta y 70% train, 30% temporal test

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
)  

# Del 30% temporal, mitad validation y mitad test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# Guardamos una copia para revisar el antes y el despues de baños
X_train_before = X_train.copy()
mask_banos_train = X_train_before["Baños"].isna()

# KNN fit solo en train
imputer_knn = KNNImputer(n_neighbors=5)
imputer_knn.fit(X_train[cols_all])

# Transform por separado
X_train.loc[:, cols_all] = imputer_knn.transform(X_train[cols_all])
X_val.loc[:, cols_all]   = imputer_knn.transform(X_val[cols_all])
X_test.loc[:, cols_all]  = imputer_knn.transform(X_test[cols_all])

# Redondear enteros
X_train[cols_int] = X_train[cols_int].round(0).astype(int)
X_val[cols_int]   = X_val[cols_int].round(0).astype(int)
X_test[cols_int]  = X_test[cols_int].round(0).astype(int)

# Revisamos como quedaron los registros cuyo valor de Baños estaba vacío
comparacion_banos = pd.DataFrame({
    "Tipo de Inmueble": X_train_before.loc[mask_banos_train, "Tipo de Inmueble"],
    "Area_antes": X_train_before.loc[mask_banos_train, "Área Construida (m2)"],
    "Habitaciones_antes": X_train_before.loc[mask_banos_train, "Habitaciones"],
    "Baños_antes": X_train_before.loc[mask_banos_train, "Baños"],
    "Area_despues": X_train.loc[mask_banos_train, "Área Construida (m2)"],
    "Habitaciones_despues": X_train.loc[mask_banos_train, "Habitaciones"],
    "Baños_despues": X_train.loc[mask_banos_train, "Baños"],
})

display(comparacion_banos.sort_values(["Baños_despues", "Area_despues"], ascending=[False, True]).head(20))

# Checamos el shape de las variables de entreno y prueba X, Y
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

,Tipo de Inmueble,Area_antes,Habitaciones_antes,Baños_antes,Area_despues,Habitaciones_despues,Baños_despues
86,Casa,580.0,NaN,NaN,580.0,7,8
5536,Casa,300.0,8.0,NaN,300.0,8,6
1965,Casa,580.0,14.0,NaN,580.0,14,6
4753,Casa,205.0,3.0,NaN,205.0,3,4
3359,Apartamento,144.0,3.0,NaN,144.0,3,3
5020,Casa,158.0,NaN,NaN,158.0,3,3
3595,Apartamento,168.0,NaN,NaN,168.0,4,3
5026,Casa,190.0,NaN,NaN,190.0,4,3
4777,Casa,250.0,NaN,NaN,250.0,4,3
5247,Apartaestudio,52.0,1.0,NaN,52.0,1,2


X_train: (3719, 10)
y_train: (3719,)
X_val:   (797, 10)
y_val:   (797,)
X_test:  (798, 10)
y_test:  (798,)


In [41]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

cols_all = ["Baños", "Habitaciones", "Área Construida (m2)"]

fig = make_subplots(rows=1, cols=len(cols_all), subplot_titles=cols_all)

for i, col in enumerate(cols_all, start=1):
    fig.add_trace(
        go.Box(y=X_train[col], name=col, showlegend=False),
        row=1, col=i
    )

fig.update_layout(title="Distribución de variables imputadas con KNN", 
                  height=400, width=300 * len(cols_all))
fig.show()

<p style="color:red;"><strong>Cambio:</strong> se movió la imputación KNN al flujo de entrenamiento, se agregó una comparación antes/después para <code>Baños</code> y la visualización ahora usa solo las columnas realmente imputadas con KNN.</p>

## **Pipeline: preprocesamiento + modelo (evitar data leakage)**

In [43]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5314 entries, 0 to 5611
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    5314 non-null   int64  
 1   Barrio                5314 non-null   object 
 2   Tipo de Inmueble      5314 non-null   object 
 3   Estado                5314 non-null   object 
 4   Antigüedad            5314 non-null   object 
 5   Área Construida (m2)  5314 non-null   float64
 6   Área Privada (m2)     5314 non-null   float64
 7   Estrato               5314 non-null   float64
 8   Baños                 5273 non-null   float64
 9   Habitaciones          5226 non-null   float64
 10  Parqueaderos          5314 non-null   int64  
 11  Piso N°               2738 non-null   float64
 12  URL                   5314 non-null   object 
 13  Precio                5314 non-null   int64  
 14  Barrio_clean          5314 non-null   object 
 15  Barrio_group          5314

### **Pipelines y transformaciones**

Modelos: 

1. LinearRegression
2. RandomForestRegressor
3. LGBMRegressor

- `PowerTransformer` para normalizar la distribución del Precio, usamos `Yeo-Johnson` porque a pesar de que nuestra variable `Precio` ya tiene valores positivos y podríamos usar `Box-Cox`, nos pareció mejor un metodo que fuera tan estricto, es mas flexible, y aunque ambos buscan redcir la asimetría y estabilizar la varianza, Box-Cox es estricamente mayores a 0, y queríamos una transformación mas segura y fácil de integrar en la pipeline

ademas, es mas flexible que hacer escala logarítimca `np.log1p` sobre nuestra y "precio" porque se adapta a los datos en vez de asumir siempre logartimo.

- PowerTransformer(y)  →  normaliza la distribución del PRECIO 
- RobustScaler(X)      →  escala numeric_features siendo robusto a outliers solo para LinearRegression porque en los de arboles el escalado no influye

Para el modelo lineal (`LinearRegression`) se usa **`RobustScaler`** en lugar de `StandardScaler`.

| Scaler | Fórmula | Problema |
|---|---|---|
| `StandardScaler` | `z = (x − media) / std` | La media y std se ven jaladas por outliers |
| `RobustScaler` | `z = (x − mediana) / IQR` | La mediana y el IQR son resistentes a outliers |

IQR (Rango Intercuartílico)** es la distancia entre el percentil 25 (Q1) y el percentil 75 (Q3).
Cubre el **50% central de los datos**, ignorando los extremos al calcular la escala.

In [44]:
areas = ["Área Construida (m2)", "Área Privada (m2)"]

df_filtrado[areas].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Área Construida (m2),5314.0,106.319407,103.569580,1.0,20.0,35.0,60.0,80.0,115.0,280.0,480.0,2416.0
Área Privada (m2),5314.0,106.755237,106.914379,1.0,20.0,35.0,60.0,79.0,115.0,280.0,500.0,2416.0


<span style="color: orange;"><strong>Observación actualizada</strong></span>

`Área Privada` aún presenta valores máximos muy altos, por lo que el filtro debe aplicarse en ambas variables de área:

```python
df_filtrado = df_filtrado[
    (df_filtrado["Área Construida (m2)"] <= 2500) &
    (df_filtrado["Área Privada (m2)"] <= 2500)
].copy()
```


Aunque el filtro reduce bastante los valores extremos, la distribución de las áreas sigue siendo asimétrica.

Mediana de Área Construida (m2): 80 m²
Media: 106.7 m²
IQR: 115 - 60 = 55 m²

Esto sugiere que todavía hay valores altos que desplazan la media hacia arriba.
Por eso, StandardScaler seguiría centrando con respecto a una media afectada por extremos, mientras que RobustScaler usa la mediana y el rango intercuartílico (IQR), representando mejor el comportamiento típico de los inmuebles.

In [45]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Área Construida (m2)", "Área Privada (m2)")
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Construida (m2)"],
        name="Área Construida",
        marker=dict(color="#0f766e"),
        fillcolor="rgba(15, 118, 110, 0.35)",
        line=dict(color="#0f766e"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área construida: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Privada (m2)"],
        name="Área Privada",
        marker=dict(color="#b45309"),
        fillcolor="rgba(180, 83, 9, 0.35)",
        line=dict(color="#b45309"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área privada: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Distribución y valores extremos en las variables de área",
    template="plotly_white",
    showlegend=False,
    width=1050,
    height=450,
    font=dict(size=13),
    margin=dict(t=70, l=40, r=40, b=40)
)

fig.update_xaxes(title_text="Metros cuadrados", row=1, col=1)
fig.update_xaxes(title_text="Metros cuadrados", row=1, col=2)

fig.show()


In [46]:
# Mira qué hay entre 500 y 4000
df[(df["Área Construida (m2)"] > 500) & 
   (df["Área Construida (m2)"] <= 4000)][["Barrio", "Tipo de Inmueble", "Área Construida (m2)", "Precio"]].sort_values("Área Construida (m2)", ascending=False).head(20)

,Barrio,Tipo de Inmueble,Área Construida (m2),Precio
206,Castilla,Apartamento,4000.0,1000000
1480,Buenos aires,Apartamento,4000.0,1150000
215,Castilla,Apartamento,4000.0,950000
254,Pedregal,Apartamento,4000.0,800000
217,Girardot,Apartaestudio,4000.0,1100000
703,Boston,Apartamento,3900.0,830000
704,Boston,Apartamento,3900.0,830000
705,Boston,Apartamento,3900.0,830000
1328,Buenos aires,Apartamento,3800.0,1200000
2166,Centro,Apartaestudio,3700.0,1100000


In [47]:
df_filtrado['Barrio_group'].value_counts()

Barrio_group
EL POBLADO          890
LAURELES            611
BELEN               288
CALASANZ            251
OTROS               205
                   ... 
MANRIQUE CENTRAL      4
LAS GRANJAS           4
BARRIO CRISTOBAL      4
GRANIZAL              2
ALFONSO LOPEZ         1
Name: count, Length: 115, dtype: int64

In [55]:
# Transformers

# ---------------------------Transformadores---------------------------

numeric_transformer_lr = Pipeline(
    steps=[
       ("scaler", RobustScaler()),  # escalado para modelos lineales
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", TargetEncoder(smooth=10)),  # importante para barrios con pocas muestras, suaviza la media del target para evitar overfitting, toma el promedio global
    ]
)

# Usamos TargetEncoder porque resume las categorías según su relación con el target ("Precio"),
# y evita expandir demasiado la dimensionalidad cuando hay varias categorías.

# ---------------------------Preprocesadores---------------------------

# ---Modelo lineal---
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---Trees: RandomForest y LightGBM---
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------Pipelines---------------------------

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", lr_preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LGBMRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)


# **Entrenar: levantamos el MLflow Tracking Server**

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```
Opción que me funcionó con Powershell
```powershell
mlflow server `
  --host 127.0.0.1 `
  --port 5001 `
  --backend-store-uri sqlite:///mlflow.db `
  --default-artifact-root ./mlruns
```

`Puerto: http://127.0.0.1:5001`

Y se crea el file `mlflow.db`

In [56]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5001'


In [57]:
for col in categorical_features:
    counts = X_train[col].value_counts()
    raros = counts[counts == 1]
    if len(raros) > 0:
        print(f"{col}: {raros.index.tolist()}")

Barrio_group: ['ALFONSO LOPEZ', 'GRANIZAL']


# **LinearRegression**


In [59]:

mlflow.set_experiment("proyecto2-linear-regression")

with mlflow.start_run(run_name="baseline_linear_regression"):
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "linear_regression")
    mlflow.set_tag("target", "Precio")
    mlflow.set_tag("features", ",".join(features))

    lr_pipeline.fit(X_train, y_train)

    y_pred_train = lr_pipeline.predict(X_train)
    y_pred_val = lr_pipeline.predict(X_val)
    y_pred_test = lr_pipeline.predict(X_test)

    print("NaN en y_pred_train:", np.isnan(y_pred_train).sum())
    print("NaN en y_pred_val:", np.isnan(y_pred_val).sum())
    print("NaN en y_pred_test:", np.isnan(y_pred_test).sum())
    print(y_pred_train[:10])



    train_mae = mean_absolute_error(y_train, y_pred_train)
    train_rmse = mean_squared_error(y_train, y_pred_train) ** 0.5
    train_r2 = r2_score(y_train, y_pred_train)

    val_mae = mean_absolute_error(y_val, y_pred_val)
    val_rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
    val_r2 = r2_score(y_val, y_pred_val)

    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("numeric_features", numeric_features)
    mlflow.log_param("categorical_features", categorical_features)

    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_r2", val_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(lr_pipeline, "model")

print("LinearRegression")
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

print("=" * 40)
print("LinearRegression")
print("=" * 40)
print(f"Train      → RMSE: ${train_rmse:,.0f}  R²: {train_r2:.4f}")
print(f"Validation → RMSE: ${val_rmse:,.0f}  R²: {val_r2:.4f}")
print(f"Test       → RMSE: ${test_rmse:,.0f}  MAE: ${test_mae:,.0f}  R²: {test_r2:.4f}")


c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


NaN en y_pred_train: 0
NaN en y_pred_val: 0
NaN en y_pred_test: 0
[1683514.62541246 2913905.69635878 4831246.36868734 3722360.26035156
 3183885.2200155  2877829.78249421 5260694.0453685  2591877.45514035
  887226.72403096 2204804.44442212]


2026/05/11 14:18:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 14:18:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 14:18:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run baseline_linear_regression at: http://127.0.0.1:5001/#/experiments/1/runs/17417cb19ebf4da69f0a86c2841de7f4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
LinearRegression
Train RMSE: 2275615.6496519498
Validation RMSE: 2669044.5377242193
Test RMSE: 2510804.266358651
Test MAE: 1176332.0818708388
Test R2: 0.5510977547815363
LinearRegression
Train      → RMSE: $2,275,616  R²: 0.6847
Validation → RMSE: $2,669,045  R²: 0.5342
Test       → RMSE: $2,510,804  MAE: $1,176,332  R²: 0.5511


# **RandomForestRegressor**

In [60]:

mlflow.set_experiment("proyecto2-random-forest-optuna")
mlflow.autolog(log_models=False)

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }

    rf_pipeline_optuna = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", TransformedTargetRegressor(
                regressor=RandomForestRegressor(**params),
                transformer=PowerTransformer(method="yeo-johnson")
            ))
        ]
    )

    with mlflow.start_run(run_name=f"rf_trial_{trial.number}", nested=True) as run:
        trial.set_user_attr("mlflow_run_id", run.info.run_id)

        mlflow.set_tag("problem_type", "regression")
        mlflow.set_tag("model_family", "random_forest")
        mlflow.set_tag("target", "Precio")
        mlflow.set_tag("optimization", "optuna")
        mlflow.set_tag("features", ",".join(features))

        rf_pipeline_optuna.fit(X_train, y_train)
        y_pred_val = rf_pipeline_optuna.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred_val)
        rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
        r2 = r2_score(y_val, y_pred_val)

        mlflow.log_metric("val_mae", mae)
        mlflow.log_metric("val_rmse", rmse)
        mlflow.log_metric("val_r2", r2)

        return rmse


2026/05/11 14:19:15 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/05/11 14:19:17 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [61]:
study_rf = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="optuna_study_random_forest") as parent_run:
    mlflow.set_tag("stage", "hpo")

    study_rf.optimize(objective_rf, n_trials=10)

    top_trials = sorted(
        [t for t in study_rf.trials if t.value is not None],
        key=lambda t: t.value
    )[:5]

    top_trials_data = [
        {
            "trial_number": t.number,
            "rmse": t.value,
            "params": t.params,
            "mlflow_run_id": t.user_attrs.get("mlflow_run_id"),
        }
        for t in top_trials
    ]

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_val_rmse", study_rf.best_value)
    mlflow.log_dict(top_trials_data, "optuna_top_trials_rf.json")
    mlflow.log_dict(study_rf.best_params, "optuna_best_params_rf.json")

print(f"Parent run id: {parent_run.info.run_id}")
print(f"Best params: {study_rf.best_params}")
print(f"Best validation RMSE: {study_rf.best_value}")


[I 2026-05-11 14:19:20,109] A new study created in memory with name: no-name-be63c882-7eac-4b40-a03e-4e5f198dec6d
2026/05/11 14:19:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\

🏃 View run rf_trial_0 at: http://127.0.0.1:5001/#/experiments/2/runs/4cb14b8445af46a9a41e0c28422bc264
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:19:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_1 at: http://127.0.0.1:5001/#/experiments/2/runs/c1bf2a27ff104d089260a0498e36e901
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:19:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

...

🏃 View run rf_trial_7 at: http://127.0.0.1:5001/#/experiments/2/runs/bfc5cb878f5941029886cd80a9e50b00
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:22:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_8 at: http://127.0.0.1:5001/#/experiments/2/runs/f90b54dd8a384d76aa8eb10e4b8d6dc5
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:22:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_9 at: http://127.0.0.1:5001/#/experiments/2/runs/0171e087c15943b29422fde82bfc818a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
🏃 View run optuna_study_random_forest at: http://127.0.0.1:5001/#/experiments/2/runs/295d91bca0ea45f9a2218d282966cd9f
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
Parent run id: 295d91bca0ea45f9a2218d282966cd9f
Best params: {'n_estimators': 464, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': None}
Best validation RMSE: 1851044.5055761028


In [62]:
# Evaluación final del mejor RandomForest en test

best_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_rf_pipeline.fit(X_train, y_train)

y_pred_test_rf = best_rf_pipeline.predict(X_test)

test_mae_rf = mean_absolute_error(y_test, y_pred_test_rf)
test_rmse_rf = mean_squared_error(y_test, y_pred_test_rf) ** 0.5
test_r2_rf = r2_score(y_test, y_pred_test_rf)

print("RandomForestRegressor")
print("Test MAE:", test_mae_rf)
print("Test RMSE:", test_rmse_rf)
print("Test R2:", test_r2_rf)


2026/05/11 14:23:10 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0885c23237174aa59ec09514a5853f2f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/11 14:23:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling

🏃 View run rare-eel-938 at: http://127.0.0.1:5001/#/experiments/2/runs/0885c23237174aa59ec09514a5853f2f
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:25:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


RandomForestRegressor
Test MAE: 871406.7080610695
Test RMSE: 1716677.090143088
Test R2: 0.790152748994938


In [63]:
print(f"Best trial: {study_rf.best_trial.number}")
print(f"Best value (Validation RMSE): {study_rf.best_value:.4f}")
print("Best params:")
for key, value in study_rf.best_params.items():
    print(f"  {key}: {value}")

best_params_rf = {**study_rf.best_params, "random_state": 42, "n_jobs": -1}

best_pipeline_rf = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(**best_params_rf),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_pipeline_rf.fit(X_train, y_train)

y_pred_train_rf = best_pipeline_rf.predict(X_train)
y_pred_val_rf = best_pipeline_rf.predict(X_val)
y_pred_test_rf = best_pipeline_rf.predict(X_test)

train_mae_rf = mean_absolute_error(y_train, y_pred_train_rf)
train_rmse_rf = mean_squared_error(y_train, y_pred_train_rf) ** 0.5
train_r2_rf = r2_score(y_train, y_pred_train_rf)

val_mae_rf = mean_absolute_error(y_val, y_pred_val_rf)
val_rmse_rf = mean_squared_error(y_val, y_pred_val_rf) ** 0.5
val_r2_rf = r2_score(y_val, y_pred_val_rf)

test_mae_rf = mean_absolute_error(y_test, y_pred_test_rf)
test_rmse_rf = mean_squared_error(y_test, y_pred_test_rf) ** 0.5
test_r2_rf = r2_score(y_test, y_pred_test_rf)

print("\nRandomForestRegressor - mejor modelo")
print(f"Train MAE: {train_mae_rf:.4f}")
print(f"Train RMSE: {train_rmse_rf:.4f}")
print(f"Train R2: {train_r2_rf:.4f}")

print(f"\nValidation MAE: {val_mae_rf:.4f}")
print(f"Validation RMSE: {val_rmse_rf:.4f}")
print(f"Validation R2: {val_r2_rf:.4f}")

print("\n--- Métricas finales para comparar modelos ---")
print(f"Test MAE: {test_mae_rf:.4f}")
print(f"Test RMSE: {test_rmse_rf:.4f}")
print(f"Test R2: {test_r2_rf:.4f}")


Best trial: 5
Best value (Validation RMSE): 1851044.5056
Best params:
  n_estimators: 464
  max_depth: 15
  min_samples_split: 3
  min_samples_leaf: 2
  max_features: None


2026/05/11 14:26:15 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0857d6819d5c49dbb2b65bb8d5fc4606', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/11 14:26:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling

🏃 View run sneaky-crow-1000 at: http://127.0.0.1:5001/#/experiments/2/runs/0857d6819d5c49dbb2b65bb8d5fc4606
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/11 14:28:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/11 14:28:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDr


RandomForestRegressor - mejor modelo
Train MAE: 746177.0275
Train RMSE: 1807540.3432
Train R2: 0.8011

Validation MAE: 927815.4949
Validation RMSE: 1879075.0110
Validation R2: 0.7691

--- Métricas finales para comparar modelos ---
Test MAE: 884789.7248
Test RMSE: 1748718.1069
Test R2: 0.7822


# **LGBMRegressor**

In [64]:
mlflow.set_experiment("proyecto2-lightgbm-optuna")
mlflow.autolog(log_models=False)

def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "verbosity": -1,
    }

    lgbm_pipeline_optuna = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", TransformedTargetRegressor(
                regressor=LGBMRegressor(**params),
                transformer=PowerTransformer(method="yeo-johnson")
            ))
        ]
    )

    with mlflow.start_run(run_name=f"lgbm_trial_{trial.number}", nested=True) as run:
        trial.set_user_attr("mlflow_run_id", run.info.run_id)

        mlflow.set_tag("problem_type", "regression")
        mlflow.set_tag("model_family", "lightgbm")
        mlflow.set_tag("target", "Precio")
        mlflow.set_tag("optimization", "optuna")
        mlflow.set_tag("features", ",".join(features))

        lgbm_pipeline_optuna.fit(X_train, y_train)
        y_pred_val = lgbm_pipeline_optuna.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred_val)
        rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
        r2 = r2_score(y_val, y_pred_val)

        mlflow.log_metric("val_mae", mae)
        mlflow.log_metric("val_rmse", rmse)
        mlflow.log_metric("val_r2", r2)

        return rmse


2026/05/11 14:29:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/05/11 14:29:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [65]:
study_lgbm = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="optuna_study_lightgbm") as parent_run:
    mlflow.set_tag("stage", "hpo")

    study_lgbm.optimize(objective_lgbm, n_trials=10)

    top_trials = sorted(
        [t for t in study_lgbm.trials if t.value is not None],
        key=lambda t: t.value
    )[:5]

    top_trials_data = [
        {
            "trial_number": t.number,
            "rmse": t.value,
            "params": t.params,
            "mlflow_run_id": t.user_attrs.get("mlflow_run_id"),
        }
        for t in top_trials
    ]

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("best_val_rmse", study_lgbm.best_value)
    mlflow.log_dict(top_trials_data, "optuna_top_trials_lgbm.json")
    mlflow.log_dict(study_lgbm.best_params, "optuna_best_params_lgbm.json")

print(f"Parent run id: {parent_run.info.run_id}")
print(f"Best params: {study_lgbm.best_params}")
print(f"Best validation RMSE: {study_lgbm.best_value}")


[I 2026-05-11 14:29:59,652] A new study created in memory with name: no-name-b98e56c1-46fd-4cc4-a3c8-56ea39267a87
2026/05/11 14:30:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\

🏃 View run lgbm_trial_0 at: http://127.0.0.1:5001/#/experiments/3/runs/f376c748f73949fcbac10f93eaa14b6c
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/11 14:30:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_1 at: http://127.0.0.1:5001/#/experiments/3/runs/90249ac2e41c49a09a284278ad33343c
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/11 14:30:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

...

🏃 View run lgbm_trial_7 at: http://127.0.0.1:5001/#/experiments/3/runs/cc6aa774a5c04c48a4d3036f3eef2a86
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/11 14:31:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_8 at: http://127.0.0.1:5001/#/experiments/3/runs/1337f9cddcb84d78b8df9c48b1237b07
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/11 14:31:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_9 at: http://127.0.0.1:5001/#/experiments/3/runs/8a715810fd2543f5aaadbe880854c785
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
🏃 View run optuna_study_lightgbm at: http://127.0.0.1:5001/#/experiments/3/runs/c7ab3766e15e4b769b8bf3036becda08
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
Parent run id: c7ab3766e15e4b769b8bf3036becda08
Best params: {'n_estimators': 435, 'learning_rate': 0.07174702728560882, 'num_leaves': 60, 'max_depth': 10, 'min_child_samples': 18, 'subsample': 0.9740612171227474, 'colsample_bytree': 0.8356825894212427}
Best validation RMSE: 1734785.3970953212


In [66]:

print(f"Best trial: {study_lgbm.best_trial.number}")
print(f"Best value (Validation RMSE): {study_lgbm.best_value:.4f}")
print("Best params:")
for key, value in study_lgbm.best_params.items():
    print(f"  {key}: {value}")

best_params_lgbm = {**study_lgbm.best_params, "random_state": 42, "verbosity": -1}

best_pipeline_lgbm = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LGBMRegressor(**best_params_lgbm),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_pipeline_lgbm.fit(X_train, y_train)

y_pred_train_lgbm = best_pipeline_lgbm.predict(X_train)
y_pred_val_lgbm = best_pipeline_lgbm.predict(X_val)
y_pred_test_lgbm = best_pipeline_lgbm.predict(X_test)

train_mae_lgbm = mean_absolute_error(y_train, y_pred_train_lgbm)
train_rmse_lgbm = mean_squared_error(y_train, y_pred_train_lgbm) ** 0.5
train_r2_lgbm = r2_score(y_train, y_pred_train_lgbm)

val_mae_lgbm = mean_absolute_error(y_val, y_pred_val_lgbm)
val_rmse_lgbm = mean_squared_error(y_val, y_pred_val_lgbm) ** 0.5
val_r2_lgbm = r2_score(y_val, y_pred_val_lgbm)

test_mae_lgbm = mean_absolute_error(y_test, y_pred_test_lgbm)
test_rmse_lgbm = mean_squared_error(y_test, y_pred_test_lgbm) ** 0.5
test_r2_lgbm = r2_score(y_test, y_pred_test_lgbm)

print("\nLGBMRegressor - mejor modelo")
print(f"Train MAE: {train_mae_lgbm:.4f}")
print(f"Train RMSE: {train_rmse_lgbm:.4f}")
print(f"Train R2: {train_r2_lgbm:.4f}")

print(f"\nValidation MAE: {val_mae_lgbm:.4f}")
print(f"Validation RMSE: {val_rmse_lgbm:.4f}")
print(f"Validation R2: {val_r2_lgbm:.4f}")

print("\n--- Métricas finales para comparar modelos ---")
print(f"Test MAE: {test_mae_lgbm:.4f}")
print(f"Test RMSE: {test_rmse_lgbm:.4f}")
print(f"Test R2: {test_r2_lgbm:.4f}")


2026/05/11 14:32:09 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'a8f1fff76c2540b68ee469f2abb36f6d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


Best trial: 7
Best value (Validation RMSE): 1734785.3971
Best params:
  n_estimators: 435
  learning_rate: 0.07174702728560882
  num_leaves: 60
  max_depth: 10
  min_child_samples: 18
  subsample: 0.9740612171227474
  colsample_bytree: 0.8356825894212427


2026/05/11 14:32:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run redolent-fawn-491 at: http://127.0.0.1:5001/#/experiments/3/runs/a8f1fff76c2540b68ee469f2abb36f6d
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/05/11 14:32:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <htt


LGBMRegressor - mejor modelo
Train MAE: 700352.7167
Train RMSE: 1614879.0165
Train R2: 0.8412

Validation MAE: 842549.2009
Validation RMSE: 1712402.2133
Validation R2: 0.8083

--- Métricas finales para comparar modelos ---
Test MAE: 840711.2025
Test RMSE: 1581699.6013
Test R2: 0.8219


In [70]:
resultados_finales = pd.DataFrame({
    "Modelo": ["Ridge", "RandomForestRegressor", "LGBMRegressor"],
    "Test MAE": [test_mae, test_mae_rf, test_mae_lgbm],
    "Test RMSE": [test_rmse, test_rmse_rf, test_rmse_lgbm],
    "Test R2": [test_r2, test_r2_rf, test_r2_lgbm],
})

resultados_finales = resultados_finales.sort_values("Test RMSE").reset_index(drop=True)
resultados_finales

,Modelo,Test MAE,Test RMSE,Test R2
0,LGBMRegressor,8.407112e+05,1.581700e+06,0.821855
1,RandomForestRegressor,8.847897e+05,1.748718e+06,0.782246
2,Ridge,1.176332e+06,2.510804e+06,0.551098


In [71]:
# Formatear para presentación
resultados_finales["Test MAE"] = resultados_finales["Test MAE"].apply(lambda x: f"${x:,.0f}")
resultados_finales["Test RMSE"] = resultados_finales["Test RMSE"].apply(lambda x: f"${x:,.0f}")
resultados_finales["Test R2"] = resultados_finales["Test R2"].apply(lambda x: f"{x:.4f}")

resultados_finales

,Modelo,Test MAE,Test RMSE,Test R2
0,LGBMRegressor,"$840,711","$1,581,700",0.8219
1,RandomForestRegressor,"$884,790","$1,748,718",0.7822
2,Ridge,"$1,176,332","$2,510,804",0.5511
